# 🎙️ Indic-Speak Bengali Text-to-Speech (Dual-GPU Kaggle Edition)

High-throughput parallel Bengali speech synthesis powered by **Bodhan AI's Indic-Speak (3.3B)** and Vocos neural vocoder.

### ⚡ Dual-GPU Parallelism (2x Speedup):
This notebook is optimized for Kaggle's **GPU T4 x 2** accelerator. It distributes text chunks across **both Tesla T4 GPUs (`cuda:0` and `cuda:1`) simultaneously**, halving generation time!

---

## ⚙️ Essential Kaggle Settings
In Kaggle's right sidebar (**Settings** panel):
1. **Accelerator:** Select **GPU T4 x 2** (or GPU P100).
2. **Internet:** Switch **Internet to ON** (required to load weights from Hugging Face).

---

## 📂 Providing Input Files
Upload your Bengali `.md` or `.txt` files either by:
* Dragging & dropping them into the `/kaggle/working/` folder in the left file tree, OR
* Clicking **Add Data** in the top right to attach a Kaggle Dataset containing your documents.

In [ ]:
# Cell 1: Environment Setup & Hardware Check
import os
import sys
import subprocess
import torch

print('=' * 65)
print('🚀 Checking GPU Accelerators...')
num_gpus = torch.cuda.device_count()
if num_gpus > 1:
    print(f'✔ DUAL GPUs DETECTED: {num_gpus} GPUs available for parallel synthesis!')
    for i in range(num_gpus):
        name = torch.cuda.get_device_name(i)
        vram = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f'   • cuda:{i} -> {name} ({vram:.1f} GB VRAM)')
elif num_gpus == 1:
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'✔ Single GPU detected: {name} ({vram:.1f} GB VRAM)')
    print('💡 Tip: For 2x faster synthesis, switch Accelerator to "GPU T4 x 2" in Notebook Settings!')
else:
    print('⚠️ No GPU detected! Please enable GPU in Kaggle (Sidebar -> Settings -> Accelerator -> GPU T4 x 2).')
print('=' * 65)

print('📦 Installing neural TTS & audio dependencies...')
!pip install -q "transformers>=5" snac soundfile accelerate tqdm

os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
print('✔ Environment setup complete!')

In [ ]:
# Cell 2: Hugging Face Authentication & Model Snapshot Download
import os
from pathlib import Path
from huggingface_hub import login, snapshot_download

# Hugging Face Access Token
# 1. Automatically loaded from Kaggle Secrets (Add-ons -> Secrets -> Key: HF_TOKEN)
# 2. Or from environment variable HF_TOKEN
# 3. Or prompted interactively if not set
HF_TOKEN = os.environ.get("HF_TOKEN", "")

# Check Kaggle User Secrets
try:
    from kaggle_secrets import UserSecretsClient
    secret_token = UserSecretsClient().get_secret("HF_TOKEN")
    if secret_token:
        HF_TOKEN = secret_token
        print("✔ Loaded HF_TOKEN from Kaggle Secrets.")
except Exception:
    pass

if not HF_TOKEN:
    import getpass
    print("👉 Enter your Hugging Face Access Token below:")
    try:
        HF_TOKEN = getpass.getpass("HF_TOKEN: ").strip()
    except Exception:
        HF_TOKEN = input("HF_TOKEN: ").strip()

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN is required to download bodhan-ai/indic-speak. "
        "Please provide your token via Kaggle Secrets (HF_TOKEN) or paste it when prompted."
    )

# Log in globally to Hugging Face
login(token=HF_TOKEN, add_to_git_credential=False)
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
print("✔ Authenticated with Hugging Face Hub successfully!")

MODEL_ID = "bodhan-ai/indic-speak"
MODEL_DIR = "/kaggle/working/indic_speak_model"
Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)

print(f"📥 Downloading \x27{MODEL_ID}\x27 weights into {MODEL_DIR}...")
try:
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=MODEL_DIR,
        token=HF_TOKEN,
    )
    print("✔ Model weights downloaded and cached successfully!")
except Exception as e:
    err = str(e)
    if "401" in err:
        print("\n❌ 401 UNAUTHORIZED: Check token validity at https://huggingface.co/settings/tokens")
    elif "403" in err or "restricted" in err.lower() or "gated" in err.lower():
        print("\n🔒 403 FORBIDDEN: Visit https://huggingface.co/bodhan-ai/indic-speak to accept the license.")
    raise


In [ ]:
# Cell 3: Load Model Instance(s) onto Available GPUs (Dual-GPU Supported)
import sys
import time
import torch

if MODEL_DIR not in sys.path:
    sys.path.insert(0, MODEL_DIR)

from inference import TTS

num_gpus = torch.cuda.device_count()
tts_workers = {}

print('=' * 65)
if num_gpus >= 2:
    print(f'🚀 Initializing Dual-GPU Workers (cuda:0 and cuda:1)...')
    for gpu_id in range(num_gpus):
        dev = f'cuda:{gpu_id}'
        print(f'   Loading Indic-Speak model on {dev} ({torch.cuda.get_device_name(gpu_id)})...')
        t0 = time.time()
        tts_workers[dev] = TTS(MODEL_DIR, device=dev)
        print(f'   ✔ {dev} loaded in {time.time() - t0:.1f}s.')
elif num_gpus == 1:
    dev = 'cuda:0'
    print(f'🧠 Loading Indic-Speak model on {dev} ({torch.cuda.get_device_name(0)})...')
    t0 = time.time()
    tts_workers[dev] = TTS(MODEL_DIR, device=dev)
    print(f'✔ {dev} loaded in {time.time() - t0:.1f}s.')
else:
    dev = 'cpu'
    print('⚠️ Loading on CPU (slow)...')
    tts_workers[dev] = TTS(MODEL_DIR, device=dev)

print(f'✔ Ready! {len(tts_workers)} GPU worker(s) active in memory.')
print('=' * 65)

In [ ]:
# Cell 4: Bengali Text Normalizer & Sentence Chunker
import re
import unicodedata
from typing import List

def normalize_bengali_text(text: str) -> str:
    """Normalizes Unicode (NFKC), strips markdown syntax, and standardizes punctuation."""
    if not text:
        return ''
    text = unicodedata.normalize('NFKC', text)
    # Strip markdown elements
    text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'```[\s\S]*?```', '', text)
    text = re.sub(r'`[^`]+`', '', text)
    text = re.sub(r'^\s*[#*+->]+\s*', '', text, flags=re.MULTILINE)
    text = re.sub(r'[*_~]+', '', text)
    
    # Bengali punctuation
    text = text.replace('||', '॥').replace('|', '।')
    text = re.sub(r'\?+', '?', text)
    text = re.sub(r'!+', '!', text)
    text = re.sub(r'।+', '।', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def chunk_bengali_text(text: str, max_chunk_len: int = 350) -> List[str]:
    """
    Intelligently chunks Bengali text by sentence delimiters ('।', '॥', '?', '!', '\n').
    Groups consecutive sentences up to max_chunk_len (~20s of speech).
    """
    clean = normalize_bengali_text(text)
    if not clean:
        return []
    
    pattern = r'([^।॥?!\n]+[।॥?!]?)'
    raw_segments = re.findall(pattern, clean)
    sentences = [s.strip() for s in raw_segments if s.strip()]
    
    chunks = []
    current = ''
    for s in sentences:
        if len(s) > max_chunk_len:
            if current:
                chunks.append(current.strip())
                current = ''
            parts = re.split(r'([,;:\-–—]\s*)', s)
            sub_curr = ''
            for p in parts:
                if len(sub_curr) + len(p) <= max_chunk_len:
                    sub_curr += p
                else:
                    if sub_curr.strip():
                        chunks.append(sub_curr.strip())
                    sub_curr = p
            if sub_curr.strip():
                chunks.append(sub_curr.strip())
            continue
            
        candidate = f'{current} {s}'.strip() if current else s
        if len(candidate) <= max_chunk_len:
            current = candidate
        else:
            if current:
                chunks.append(current.strip())
            current = s
            
    if current:
        chunks.append(current.strip())
        
    return [c for c in chunks if c]

print('✔ Bengali text normalizer & sentence chunker ready!')

In [ ]:
# Cell 5: Parallel Dual-GPU Synthesis Engine with Checkpointing
import os
import shutil
import subprocess
import numpy as np
import soundfile as sf
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

SAMPLE_RATE = 24000

def _synthesize_chunk_task(chunk_idx, chunk_text, device, speaker, style, chk_dir):
    """Worker task executed on assigned GPU device."""
    chunk_file = chk_dir / f'chunk_{chunk_idx:04d}.npy'
    
    # Load from cache if already generated (resumable)
    if chunk_file.exists() and chunk_file.stat().st_size > 0:
        try:
            return chunk_idx, np.load(chunk_file)
        except Exception:
            pass
            
    tts_model = tts_workers[device]
    wav = tts_model(
        text=chunk_text,
        speaker=speaker,
        style=style,
        temperature=0.6,
        top_p=0.9,
        top_k=50,
        max_new_tokens=2520,
    )
    if hasattr(wav, 'cpu'):
        wav_np = wav.cpu().numpy()
    elif hasattr(wav, 'numpy'):
        wav_np = wav.numpy()
    else:
        wav_np = np.array(wav, dtype=np.float32)
    wav_np = np.squeeze(wav_np)
    
    # Save checkpoint immediately
    np.save(chunk_file, wav_np)
    return chunk_idx, wav_np


def synthesize_text_to_audio(
    text: str,
    output_path: str,
    speaker: str = 'Ishita',
    style: str = 'NEUTRAL',
    chunk_size: int = 350,
    pause_duration: float = 0.25,
    output_format: str = 'opus',
    checkpoint_dir: str = '/kaggle/working/chunks',
) -> dict:
    """
    Synthesizes Bengali text into audio using parallel GPU workers.
    """
    chunks = chunk_bengali_text(text, max_chunk_len=chunk_size)
    if not chunks:
        raise ValueError('Input text has no readable content.')
        
    out_p = Path(output_path).resolve()
    out_p.parent.mkdir(parents=True, exist_ok=True)
    
    chk_dir = Path(checkpoint_dir)
    chk_dir.mkdir(parents=True, exist_ok=True)
    
    pause_samples = int(SAMPLE_RATE * pause_duration)
    silence = np.zeros(pause_samples, dtype=np.float32)
    
    devices = list(tts_workers.keys())
    mode_str = f'Dual-GPU ({devices[0]} + {devices[1]})' if len(devices) > 1 else f'Single GPU ({devices[0]})'
    print(f'\n🎙️ Synthesizing {len(chunks)} chunks [{mode_str}] | Speaker: {speaker} | Style: {style}')
    
    generated_segments = [None] * len(chunks)
    
    # Submit parallel chunk tasks across available GPU devices
    with ThreadPoolExecutor(max_workers=len(devices)) as executor:
        futures = []
        for idx, chunk in enumerate(chunks):
            assigned_device = devices[idx % len(devices)]
            f = executor.submit(
                _synthesize_chunk_task,
                chunk_idx=idx,
                chunk_text=chunk,
                device=assigned_device,
                speaker=speaker,
                style=style,
                chk_dir=chk_dir,
            )
            futures.append(f)
            
        desc_label = f'Synthesizing ({len(devices)}x GPU)'
        for f in tqdm(as_completed(futures), total=len(futures), desc=desc_label):
            c_idx, wav_np = f.result()
            generated_segments[c_idx] = wav_np
            
    # Interleave segments with pause silence
    combined = []
    for i, seg in enumerate(generated_segments):
        combined.append(seg)
        if i < len(generated_segments) - 1 and pause_samples > 0:
            combined.append(silence)
            
    full_audio = np.concatenate(combined, axis=0)
    
    # Normalize volume
    max_val = np.max(np.abs(full_audio))
    if max_val > 0.99:
        full_audio = full_audio / max_val * 0.95
        
    temp_wav = '/kaggle/working/temp_render.wav'
    sf.write(temp_wav, full_audio, samplerate=SAMPLE_RATE)
    
    # Format conversion via ffmpeg
    fmt = output_format.lower().lstrip('.')
    if fmt == 'opus':
        cmd = ['ffmpeg', '-y', '-i', temp_wav, '-c:a', 'libopus', '-b:a', '24k', '-vbr', 'on', str(out_p)]
        subprocess.run(cmd, check=True, capture_output=True)
    elif fmt == 'mp3':
        cmd = ['ffmpeg', '-y', '-i', temp_wav, '-codec:a', 'libmp3lame', '-b:a', '64k', str(out_p)]
        subprocess.run(cmd, check=True, capture_output=True)
    else:  # wav
        shutil.copyfile(temp_wav, str(out_p))
        
    if os.path.exists(temp_wav):
        os.remove(temp_wav)
        
    dur_sec = len(full_audio) / SAMPLE_RATE
    size_kb = out_p.stat().st_size / 1024
    print(f'✔ Saved: {out_p.name} ({dur_sec:.1f}s audio, {size_kb:.1f} KB)')
    
    # Cleanup temporary chunk files for this document
    shutil.rmtree(chk_dir, ignore_errors=True)
    
    return {'file': str(out_p), 'duration': dur_sec, 'size_kb': size_kb}

print('✔ Dual-GPU parallel synthesis engine ready!')

In [ ]:
# Cell 6: Batch Convert All Uploaded Files
from pathlib import Path

# ==================== USER CONFIGURATION ====================
# Voice Options: 'Ishita' (Female native Bengali, default) or 'Sourav' (Male native Bengali)
SPEAKER = 'Ishita'

# Emotion / Style: 'NEUTRAL' (default), 'HAPPY', 'SAD', 'ANGER', 'FEAR'
STYLE = 'NEUTRAL'

# Output Audio Format: 'opus' (default, superior quality & compression), 'wav', or 'mp3'
OUTPUT_FORMAT = 'opus'

# Input Search Locations (Kaggle Dataset uploads or working dir)
SEARCH_DIRS = ['/kaggle/input', '/kaggle/working']
OUTPUT_DIR = '/kaggle/working/output_audio'
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
# ============================================================

# Discover files
convertible_files = []
for sdir in SEARCH_DIRS:
    for ext in ['*.txt', '*.md', '*.markdown']:
        for fp in Path(sdir).rglob(ext):
            if 'indic_speak_model' in str(fp) or 'requirements' in fp.name or 'chunks' in str(fp):
                continue
            if fp not in convertible_files:
                convertible_files.append(fp)

print('=' * 65)
print(f'📂 Found {len(convertible_files)} Bengali document(s) to convert:')
for i, f in enumerate(convertible_files, 1):
    print(f'  {i}. {f.name} ({f})')
print('=' * 65)

if not convertible_files:
    print('\n⚠️ No .txt or .md files found!')
    print('Tip: Drag and drop your Bengali text/markdown files into /kaggle/working in the left panel, or Add Data.')
else:
    results = []
    for idx, doc_path in enumerate(convertible_files, 1):
        print(f'\n[{idx}/{len(convertible_files)}] Processing: {doc_path.name}')
        content = doc_path.read_text(encoding='utf-8', errors='replace')
        
        out_file = Path(OUTPUT_DIR) / f'{doc_path.stem}.{OUTPUT_FORMAT}'
        res = synthesize_text_to_audio(
            text=content,
            output_path=str(out_file),
            speaker=SPEAKER,
            style=STYLE,
            output_format=OUTPUT_FORMAT,
        )
        results.append(res)

    print('\n' + '=' * 65)
    print(f'🎉 Conversion Complete! Successfully converted {len(results)} file(s).')
    print('=' * 65)

In [ ]:
# Cell 7: In-Notebook Audio Player & 1-Click ZIP Download
import shutil
from pathlib import Path
import IPython.display as ipd

# 1. Play audio sample directly inside Kaggle notebook
output_files = sorted(list(Path(OUTPUT_DIR).glob(f'*.{OUTPUT_FORMAT}')))
if output_files:
    sample = output_files[0]
    print(f'🎧 Playing Sample: {sample.name}')
    ipd.display(ipd.Audio(str(sample)))

# 2. Package all generated audio into a single downloadable ZIP file
zip_base = '/kaggle/working/bengali_audio_output'
zip_path = shutil.make_archive(zip_base, 'zip', OUTPUT_DIR)
zip_size_mb = Path(zip_path).stat().st_size / (1024 * 1024)

print('\n' + '=' * 65)
print(f'📦 ZIP Archive Ready for Download:')
print(f'   📁 {zip_path} ({zip_size_mb:.2f} MB)')
print('👉 Download it from the right-hand \'Output\' tab in Kaggle!')
print('=' * 65)